# 03 · Inference & Execution-Accuracy Evaluation

Generate SQL on BIRD dev with the fine-tuned adapter, then score with
execution accuracy (EX), valid-SQL rate and exact match — plus a baseline
(same model, no fine-tuning) for a clean before/after comparison.

> Run this in the **same Colab/Kaggle session** as notebook 02 (so `outputs/`
> and `data/processed/` exist), or re-run notebook 02's SETUP + preprocessing.

In [ ]:
# --- SETUP: run me first (works on Colab, Kaggle AND locally) ---
import os, sys, subprocess
if not os.path.exists('src') and os.path.basename(os.getcwd()) != 'text2sql-finetuning':
    if not os.path.isdir('text2sql-finetuning'):
        subprocess.run(['git', 'clone', 'https://github.com/Shiverion/text2sql-finetuning.git'], check=True)
    os.chdir('text2sql-finetuning')
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())
print('src present :', os.path.isdir('src'))

## 1. Baseline — base model, zero-shot
Establishes the lift attributable to fine-tuning. (`--limit 200` keeps it
quick; remove for the full dev set.)

In [ ]:
!python -m src.inference --model_dir unsloth/Qwen2.5-Coder-1.5B-Instruct \
    --input data/processed/bird_dev.jsonl --output outputs/preds_base.jsonl --limit 200
!python -m src.evaluate --pred outputs/preds_base.jsonl --report outputs/metrics_base.json

## 2. Fine-tuned model

In [ ]:
!python -m src.inference --model_dir outputs/qwen2.5-coder-1.5b-bird-qlora \
    --input data/processed/bird_dev.jsonl --output outputs/preds_ft.jsonl --limit 200
!python -m src.evaluate --pred outputs/preds_ft.jsonl --report outputs/metrics_ft.json

## 3. Compare

In [ ]:
import json, pandas as pd, os
os.makedirs('report/figures', exist_ok=True)
base = json.load(open('outputs/metrics_base.json'))
ft   = json.load(open('outputs/metrics_ft.json'))
tbl = pd.DataFrame({'baseline': base, 'fine-tuned': ft}).loc[
    ['execution_accuracy','valid_sql_rate','exact_match']]
print(tbl)
ax = tbl.plot(kind='bar', rot=0, title='Baseline vs fine-tuned'); ax.figure.tight_layout()
ax.figure.savefig('report/figures/before_after.png', dpi=120)

## 4. Error analysis
Read the captured failures to find systematic mistakes (hallucinated columns,
wrong joins, missing GROUP BY). This is what drives the next experiment.

In [ ]:
for e in ft.get('error_samples', [])[:10]:
    print(e['type'])
    print('  pred:', e.get('pred',''))
    if 'error' in e: print('  err :', e['error'])
    print()

## 5. Try the motivating question live
Uses the bundled fintech DB, so it works even without BIRD downloaded.

In [ ]:
from src.inference import load_model, generate_batch
from src.prompts import build_messages, extract_sql
from src.schema_utils import serialize_schema
model, tok, _ = load_model('outputs/qwen2.5-coder-1.5b-bird-qlora', 2048, load_in_4bit=True)
tok.padding_side='left'; tok.pad_token = tok.pad_token or tok.eos_token
schema = serialize_schema('data/sample/db/fintech/fintech.sqlite')
q = 'Who were the top performing merchants last quarter?'
prompt = tok.apply_chat_template(build_messages(schema, q), tokenize=False, add_generation_prompt=True)
print(extract_sql(generate_batch(model, tok, [prompt], 256)[0]))